# 🩺 Sistema de Triaje Automático con IA para Neumonía Pediátrica
**Autor:** Diego Dupuy Álamo | **Arquitectura:** ResNet18 | **XAI:** Grad-CAM

---
## ⚙️ Fase 1: Pipeline de Datos y Pre-procesamiento Clínico
Extracción del dataset oficial del *Guangzhou Women and Children's Medical Center* y estandarización de las radiografías mediante transformaciones matemáticas para adaptar los tensores a la arquitectura ResNet18.

In [ ]:
import os
import shutil
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. Configuración y Descarga del Dataset
os.environ['KAGGLE_API_TOKEN'] = "KGAT_f5218b17cae974e6da5d5ea4750f45bd"
!kaggle datasets download -q -d paultimothymooney/chest-xray-pneumonia
!unzip -q -n chest-xray-pneumonia.zip -d datos_neumonia
print("✅ Fase 1.1: Base de datos clínica montada en el servidor temporal.")

# 2. Pipeline de Transformación de Imágenes
transformaciones = transforms.Compose([
    transforms.Resize((224, 224)), # Normalización geométrica espacial
    transforms.ToTensor(),         # Conversión a tensores (0 a 1)
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], # Estándar de iluminación de ImageNet
        std=[0.229, 0.224, 0.225]
    )
])

# 3. Creación de los DataLoaders (Generadores de lotes)
ruta_train = 'datos_neumonia/chest_xray/train'
ruta_val = 'datos_neumonia/chest_xray/val'

datos_train = datasets.ImageFolder(ruta_train, transform=transformaciones)
datos_val = datasets.ImageFolder(ruta_val, transform=transformaciones)

tamano_lote = 32
loader_train = DataLoader(datos_train, batch_size=tamano_lote, shuffle=True)
loader_val = DataLoader(datos_val, batch_size=tamano_lote, shuffle=False)

print(f"✅ Tubería lista: {len(datos_train)} imágenes de entrenamiento en {len(loader_train)} lotes.")

## 🧠 Fase 2: Transfer Learning y Entrenamiento del Modelo
Utilizamos los pesos pre-entrenados de ResNet18, congelando sus capas iniciales (extractores de características) y modificando el clasificador final para una detección binaria (0: Sano | 1: Neumonía). Se implementa *Early Stopping* para prevenir el sobreajuste.

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torchvision import models
import time
import copy

# 1. Configuración de Hardware
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Aceleración por hardware activa: {device.type.upper()}")

# 2. Arquitectura de Transfer Learning
modelo = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Congelamos las capas profundas para preservar el conocimiento previo
for param in modelo.parameters():
    param.requires_grad = False

# Cirugía en la capa final: Adaptamos a 2 salidas (Clasificación Binaria)
num_caracteristicas = modelo.fc.in_features
modelo.fc = nn.Linear(num_caracteristicas, 2)
modelo = modelo.to(device)

# 3. Hiperparámetros de Entrenamiento
criterio = nn.CrossEntropyLoss() # Función de pérdida clínica
optimizador = optim.Adam(modelo.fc.parameters(), lr=0.001)

epocas = 15
paciencia = 4
mejor_loss_val = float('inf')
epocas_sin_mejora = 0
mejor_modelo_pesos = copy.deepcopy(modelo.state_dict())

print("\n🚀 Iniciando Entrenamiento Clínico (Con protección Early Stopping)...")

# 4. Bucle de Entrenamiento y Validación
for epoca in range(epocas):
    inicio_tiempo = time.time()

    # -- ENTRENAMIENTO --
    modelo.train()
    perdida_train, aciertos_train, total_train = 0.0, 0, 0

    for imagenes, etiquetas in loader_train:
        imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)

        optimizador.zero_grad()
        predicciones = modelo(imagenes)
        loss = criterio(predicciones, etiquetas)
        loss.backward()
        optimizador.step()

        perdida_train += loss.item() * imagenes.size(0)
        _, pred_clase = torch.max(predicciones, 1)
        aciertos_train += torch.sum(pred_clase == etiquetas).item()
        total_train += etiquetas.size(0)

    # -- VALIDACIÓN CLÍNICA --
    modelo.eval()
    perdida_val, aciertos_val, total_val = 0.0, 0, 0

    with torch.no_grad():
        for imagenes, etiquetas in loader_val:
            imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)
            predicciones = modelo(imagenes)
            loss = criterio(predicciones, etiquetas)

            perdida_val += loss.item() * imagenes.size(0)
            _, pred_clase = torch.max(predicciones, 1)
            aciertos_val += torch.sum(pred_clase == etiquetas).item()
            total_val += etiquetas.size(0)

    # -- MÉTRICAS Y GUARDADO --
    tiempo_epoca = time.time() - inicio_tiempo
    train_loss_media = perdida_train / total_train
    train_acc = aciertos_train / total_train
    val_loss_media = perdida_val / total_val
    val_acc = aciertos_val / total_val

    print(f"Época {epoca+1}/{epocas} [{tiempo_epoca:.0f}s]") # El .0f significa "trátalo como un número con decimales (float) pero recórtalo para que muestre 0 decimales
    print(f"  Train -> Loss: {train_loss_media:.4f} | Acc: {train_acc*100:.2f}%")
    print(f"  Val   -> Loss: {val_loss_media:.4f} | Acc: {val_acc*100:.2f}%")

    if val_loss_media < mejor_loss_val:
        mejor_loss_val = val_loss_media
        mejor_modelo_pesos = copy.deepcopy(modelo.state_dict())
        torch.save(modelo.state_dict(), 'modelo_neumonia_definitivo.pth')
        epocas_sin_mejora = 0
        print("      NUEVO RÉCORD: Parámetros guardados en disco.")
    else:
        epocas_sin_mejora += 1
        print(f"   ⚠️ Sin mejora. Paciencia de Early Stopping: {epocas_sin_mejora}/{paciencia}")

    if epocas_sin_mejora >= paciencia:
        print("\n  EARLY STOPPING ACTIVADO: Se detiene el entrenamiento para evitar Overfitting.")
        break

# Restauración del modelo óptimo para auditoría
modelo.load_state_dict(mejor_modelo_pesos)
print(f"✅ Entrenamiento Finalizado. Mejor Val Loss clínico: {mejor_loss_val:.4f}")

## 📊 Fase 3: Auditoría Médica (Examen en Test Set)
Evaluación del modelo frente a 624 pacientes nunca antes vistos. Se genera una Matriz de Confusión para monitorizar métricas críticas de triaje, priorizando la Sensibilidad (Recall) para minimizar el riesgo de Falsos Negativos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# 1. Preparación del Test Set
ruta_test = 'datos_neumonia/chest_xray/test'
dataset_test = datasets.ImageFolder(ruta_test, transform=transformaciones)
loader_test = DataLoader(dataset_test, batch_size=32, shuffle=False)

# 2. Examen Ciego
modelo.eval()
etiquetas_reales = []
predicciones_modelo = []

print("Procesando 624 radiografías del conjunto de prueba...")

with torch.no_grad():
    for imagenes, etiquetas in loader_test:
        imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)
        salidas = modelo(imagenes)
        _, prediccion = torch.max(salidas, 1)

        etiquetas_reales.extend(etiquetas.cpu().numpy())
        predicciones_modelo.extend(prediccion.cpu().numpy())

# 3. Visualización de la Matriz de Confusión
cm = confusion_matrix(etiquetas_reales, predicciones_modelo)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Sano (0)', 'Neumonía (1)'],
            yticklabels=['Sano (0)', 'Neumonía (1)'])

plt.ylabel('Diagnóstico Real (Ground Truth)')
plt.xlabel('Diagnóstico IA (Predicción)')
plt.title('Matriz de Confusión - Auditoría de Triaje')
plt.show()

# 4. Reporte Clínico
print("\n%%%%%% REPORTE DE MÉTRICAS DE TRIAJE %%%%%%")
print(classification_report(etiquetas_reales, predicciones_modelo, target_names=['Sano', 'Neumonía']))

## 🔍 Fase 4: Inteligencia Artificial Explicable (Grad-CAM)
Auditoría visual de los **Falsos Negativos**. Extraemos los gradientes de la última capa convolucional (`layer4[-1]`) para generar mapas de calor y detectar sesgos algorítmicos espaciales.

In [ ]:
# Instalación de utilidades XAI
!pip install -q grad-cam

In [ ]:
import cv2
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# 1. Preparación de la Red Neuronal para Extracción de Gradientes
# Descongelamos los parámetros para permitir el retroceso de la señal matemática (Backprop)
for param in modelo.parameters():
    param.requires_grad = True

# Conectamos el "escáner" en el último bloque convolucional inteligente
target_layers = [modelo.layer4[-1]]
cam = GradCAM(model=modelo, target_layers=target_layers)

# 2. Búsqueda Activa de un Falso Negativo
modelo.eval()
imagen_fn, tensor_imagen = None, None

print("Aislando un paciente Falso Negativo para auditoría visual...")

for imagenes, etiquetas in loader_test:
    imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)
    salidas = modelo(imagenes)
    _, predicciones = torch.max(salidas, 1)

    for i in range(len(etiquetas)):
        if etiquetas[i] == 1 and predicciones[i] == 0: # Real: Neumonía | IA: Sano
            tensor_imagen = imagenes[i].unsqueeze(0)
            imagen_fn = imagenes[i].cpu()
            break

    if imagen_fn is not None:
      break

# 3. Generación del Mapa de Calor (Heatmap)
if imagen_fn is not None:
    # Des-normalización a espacio RGB (0 a 1)
    img_visual = imagen_fn.numpy().transpose(1, 2, 0)
    mean, std = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])
    img_visual = np.clip(std * img_visual + mean, 0, 1)

    # Interrogatorio a la red sobre la clase 1 (Neumonía)
    targets = [ClassifierOutputTarget(1)]
    mapa_calor_gris = cam(input_tensor=tensor_imagen, targets=targets)[0, :]
    visualizacion = show_cam_on_image(img_visual, mapa_calor_gris, use_rgb=True)

    # 4. Renderizado Final
    plt.figure(figsize=(12, 6))

    plt.subplot(1, 2, 1)
    plt.imshow(img_visual)
    plt.title("Realidad Clínica (Paciente Enfermo)")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(visualizacion)
    plt.title("Atención Algorítmica (Grad-CAM)")
    plt.axis('off')

    plt.tight_layout()
    plt.show()
    print("Resultado Auditoría: Se detecta sesgo espacial. El modelo atiende a regiones periféricas ignorando la opacidad central.")
else:
    print("No se hallaron Falsos Negativos en este lote de prueba.")